# Download Math Datasets For AVSD

This standalone notebook downloads and normalizes the math-only training set and benchmarks into Google Drive.

Output layout:
- `/content/drive/MyDrive/NLP_Project/training_set/openthought_math.jsonl`
- `/content/drive/MyDrive/NLP_Project/benchmarks/{gsm8k,aime24,aime25,hmmt25}.jsonl`

In [ ]:
# Mount Google Drive.
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ModuleNotFoundError:
    print('google.colab is unavailable; assuming Drive is already mounted or running locally.')

In [ ]:
# Install runtime dependencies. Run this in Colab before the remaining cells.
import sys
import subprocess

packages = [
    'datasets>=3.6.0',
    'tqdm',
    'pandas',
    'pyarrow',
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *packages])

In [ ]:
# Paths.
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/NLP_Project')
TRAINING_DIR = DRIVE_ROOT / 'training_set'
BENCHMARK_DIR = DRIVE_ROOT / 'benchmarks'

TRAINING_DIR.mkdir(parents=True, exist_ok=True)
BENCHMARK_DIR.mkdir(parents=True, exist_ok=True)

TRAINING_OUTPUT = TRAINING_DIR / 'openthought_math.jsonl'
BENCHMARK_OUTPUTS = {
    'gsm8k': BENCHMARK_DIR / 'gsm8k.jsonl',
    'aime24': BENCHMARK_DIR / 'aime24.jsonl',
    'aime25': BENCHMARK_DIR / 'aime25.jsonl',
    'hmmt25': BENCHMARK_DIR / 'hmmt25.jsonl',
}

print('Training output:', TRAINING_OUTPUT)
print('Benchmark dir:', BENCHMARK_DIR)

In [ ]:
# Normalization helpers.
import json
import re
from pathlib import Path
from typing import Any

from datasets import load_dataset
from tqdm.auto import tqdm


def write_jsonl(path: Path, rows: list[dict[str, Any]]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('w', encoding='utf-8') as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + '\n')


def extract_boxed_answer(text: str | None) -> str | None:
    if not text:
        return None
    start = text.rfind('\\boxed')
    if start < 0:
        return None
    brace_start = text.find('{', start)
    if brace_start < 0:
        return None
    depth = 0
    for idx in range(brace_start, len(text)):
        if text[idx] == '{':
            depth += 1
        elif text[idx] == '}':
            depth -= 1
            if depth == 0:
                return text[brace_start + 1:idx].strip()
    return None


def extract_gsm8k_answer(answer_text: str) -> str:
    if '####' in answer_text:
        return answer_text.split('####')[-1].strip()
    numbers = re.findall(r'-?\d+(?:\.\d+)?', answer_text.replace(',', ''))
    return numbers[-1] if numbers else answer_text.strip()


def compact_metadata(example: dict[str, Any], keep: tuple[str, ...]) -> dict[str, Any]:
    return {key: example[key] for key in keep if key in example}


def get_first(example: dict[str, Any], keys: tuple[str, ...], default: Any = None) -> Any:
    for key in keys:
        if key in example and example[key] is not None:
            return example[key]
    return default

In [ ]:
# Download and normalize OpenThoughts math training data.
raw = load_dataset('siyanzhao/Openthoughts_math_30k_opsd')
train_split = raw['train'] if isinstance(raw, dict) or hasattr(raw, 'keys') else raw
rows = []
for idx, ex in enumerate(tqdm(train_split, desc='OpenThoughts math')):
    problem = str(get_first(ex, ('problem', 'question', 'prompt'), '')).strip()
    solution = str(get_first(ex, ('solution', 'answer', 'response'), '')).strip()
    if not problem or not solution:
        continue
    rows.append({
        'id': str(get_first(ex, ('id', 'problem_id'), f'openthought_{idx}')),
        'source': 'siyanzhao/Openthoughts_math_30k_opsd',
        'problem': problem,
        'answer': extract_boxed_answer(solution) or '',
        'solution': solution,
        'metadata': compact_metadata(dict(ex), ('category', 'subject', 'level')),
    })

write_jsonl(TRAINING_OUTPUT, rows)
print(f'Wrote {len(rows):,} training rows to {TRAINING_OUTPUT}')

In [ ]:
# Download and normalize benchmarks.
BENCHMARK_SPECS = {
    'gsm8k': ('openai/gsm8k', 'main', 'test'),
    'aime24': ('HuggingFaceH4/aime_2024', None, 'train'),
    'aime25': ('yentinglin/aime_2025', None, 'train'),
    'hmmt25': ('MathArena/hmmt_feb_2025', None, 'train'),
}


def normalize_benchmark(name: str, dataset) -> list[dict[str, Any]]:
    rows = []
    for idx, ex in enumerate(tqdm(dataset, desc=name)):
        ex = dict(ex)
        if name == 'gsm8k':
            problem = str(ex.get('question', '')).strip()
            answer = extract_gsm8k_answer(str(ex.get('answer', '')).strip())
            original_solution = str(ex.get('answer', '')).strip()
        else:
            problem = str(get_first(ex, ('problem', 'question', 'prompt'), '')).strip()
            answer = str(get_first(ex, ('answer', 'final_answer', 'target'), '')).strip()
            original_solution = str(get_first(ex, ('solution', 'rationale'), '') or '')
        if not problem or not answer:
            continue
        rows.append({
            'id': str(get_first(ex, ('id', 'problem_id', 'problem_idx', 'question_id'), f'{name}_{idx}')),
            'source': name,
            'problem': problem,
            'answer': answer,
            'solution': original_solution,
            'metadata': compact_metadata(ex, ('url', 'year', 'contest', 'split')),
        })
    return rows

for name, (dataset_id, config, split) in BENCHMARK_SPECS.items():
    kwargs = {'split': split}
    if name in {'aime25', 'hmmt25'}:
        kwargs['trust_remote_code'] = True
    ds = load_dataset(dataset_id, config, **kwargs) if config else load_dataset(dataset_id, **kwargs)
    rows = normalize_benchmark(name, ds)
    write_jsonl(BENCHMARK_OUTPUTS[name], rows)
    print(f'Wrote {len(rows):,} {name} rows to {BENCHMARK_OUTPUTS[name]}')